In [1]:
import os
import pickle
import dill
import pprint
import itertools
import pathos
import pprint
import functools
from functools import partial
from pathlib import Path
import cProfile
import pstats

import pandas as pd
import numpy as np
from scipy import stats as st

import plotly.graph_objects as go
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

import sys
sys.path.append('/Users/leonardo.tessarolo/git/bayesian_bss/')

from src import MMSEMetropolisHastingsEstimator, MAPGradientAscentEstimator, InstantaneousMixtureModel, PosteriorContourLines, ContourLineGraphPlotter, MCMCGraphPlotter, MAPGradientAscentGraphPlotter, ExponentialPrior, LogisticSource, BayesianEstimators, TriangularSource, ExperimentExecutor

print(os.cpu_count())

np.random.seed(2000)

8


# DESCRITIVO DA IMPLEMENTAÇÃO DO CÓDIGO DO EXPERIMENTO

## i. Arquivo de entrada: execution_config

O código que executa a simulação consome um arquivo de configuração, na forma de um dicionário python. Este dicionário conterá os seguintes campos:

- 'general': informações gerais do experimento.
    - 'experiment_name': nome do experimento,
    - 'experiment_dir': diretório usado para salvar arquivos gerados,
    - 'create_folder_structure': se a estrutura de pastas deve ser criada ou já existe,
    - 'n_sources': número de fontes não observadas,
    - 'n_obs': número de observações em cada realização,
    - 'n_workers': número de executores em paralelo a serem gerados,
    - 'n_realizations': número de realizações,
    - 'A': matriz de mistura utilizada,
    - 'B': matriz de separação (inversa da matriz de misturas) utilizada,
    - 'initial_B': condição inicial de B,
    - 'normalize_posterior': se uma normalização por n_obs será feita na posteriori ou não
- 'contour': configurações da análise de curvas de nível
    - 'contour_grid_points': quantos pontos serão utilizados em cada dimensão das curvas de nível,
    - 'u_lims': limites superior e inferior da dimensão u,
    - 'v_lims':limites superior e inferior da dimensão v 
- 'map': configurações do algoritmo gradient ascent
    - 'stopping_thresh': threshold minimo de crescimento da posteriori a cada iteração para continuar otimização,        
    - 'max_it': número máximo de iterações do gradient ascent,
    - 'learning_rate': taxa de aprendizado do algoritmo
- 'mcmc': configurações do algoritmo Metropolis-Hastings        
    - 'exploration_var': variância de exploração,
    - 'n_samples': número de amostras a serem geradas,
    - 'burn_in': fração das amostras a serem descartadas para garantia de regime estacionário do processo
- 'sim': configurações de fontes, priors e casos de teste a serem executados
    - 'source_model': qual é o modelo que se está utilizando de estatística das fontes,
    - 'sources': quais diferentes fontes serão utilizadas,
    - 'priors': quais diferentes distribuições a priori serão consideradas,
    - 'test_cases': quais diferentes casos de teste (combinações de característica das fontes com priors) serão executados
    

Exemplo de arquivo execution_config:

![alt text](documentation/Captura%20de%20Tela%202025-12-14%20às%2015.22.54.png "Title")

## ii. Etapas Gerais

A sequência de ações utilizadas para executar o experimento pode ser dividida em duas etapas principais: inicialização e execução. Para ambas, utiliza-se um objeto da classe ExperimentExecutor, sendo a inicialização feita a partir do método construtor da classe ExperimentExecutor e a execução feita pelo método ExperimentExecutor.run().

![alt text](documentation/etapas_gerais.drawio.png "Title")

- Inicialização: 
    - Geração de sinais das fontes e das misturas para as diversas realizações;
    - Geração de arquivos de configuração internos do código, a partir do arquivo de configurações geral (ex: configurações individuais do MCMC e do Gradient Ascent).
    - Geração de objetos internos utilizados pelo código (ex: objeto BayesianEstimator que executa objetos MMSEMetropolisHastingsEstimator e MAPGradientAscentEstimator, que por sua vez utilizam outros objetos mais baixo-nível).
    - Entrada:
        - execution_config: arquivo de configurações geral do experimento
    - Saída:
        - execution_config: arquivo de configurações geral do experimento, enriquecido com objetos que serão utilizados para execução do experimento
        - signals: sinais das fontes e misturas, para as diferentes realizações configuradas

- Execução: 
    - Cálculo de grid para curvas de nível;
    - Execução do algoritmo Metropolis-Hastings;
    - Execução do algoritmo Gradient Ascent
    - Entrada:
        - execution_config: arquivo de configurações geral do experimento, enriquecido pós inicialização
    - Saída:
        - execution_config: arquivo de configurações geral do experimento, enriquecido com objetos que serão utilizados para execução do experimento
        - signals: sinais das fontes e misturas, para as diferentes realizações configuradas

## iii. Inicialização

![alt text](documentation/initializations.drawio.png "Title")

Descritivo funções:
- ExperimentExecutor.__initialize_signals(): gera, para cada realização, os sinais das fontes e misturas.
- ExperimentExecutor.__get_test_case_distributions(): para cada caso de teste especificado, gera objetos e configurações internas que serão utilizados para executar Gradient Ascent e MCMC.
- ExperimentExecutor.__get_exec_fns(): para cada caso de teste, gera funções a serem aplicadas para efetuar analise de curva de nivel, MCMC e gradient ascent.
- ExperimentExecutor.__save_initializations(): salva arquivo de configuração enriquecido, bem como sinais para cada realização.

## iv. Execução

![alt text](documentation/execucao.drawio.png "Title")

Descritivo funções:
- ExperimentExecutor.run(): consome arquivos signals.pkl de cada realização, bem como dicionário execution_config.pkl enriquecido após etapa setup, e executa análises de grid posteriori, amostragem MCMC e Gradient Ascent para cada realização. Função ExperimentExecutor.run() executa subfunção __run_realization(), utilizando processamento paralelo, para cada sinal das realizações.

### v. __run_realization()

Considerando uma realização r, função __run_realizations executa, para cada um dos casos de teste especificados no arquivo de configuração, análises da distribuição a posteriori.

![alt text](documentation/realization.drawio.png "Title")

Descritivo funções dentro do loop:
- bayesian_estimators_fn(): executa amostragem MCMC (Metropolis-Hastings) e Gradient Ascent, de forma a implementar estimadores MMSE e MAP, respectivamente. Instancia da função BayesianEstimators.run(), fixando argumentos segundo especificado no arquivo de configuração.
- posteriori_grid_fn(): executa cálculo de grid da distribuição a posteriori em termos dos parâmetros u e v. Utiliza objetos e métodos da classe PosteriorContourLines. fixando argumentos segundo especificado no arquivo de configuração.


In [2]:
# TODO1: representar mais precisamente que o programa nao está usando as fontes
# TODO2: acrescentar exemplos dos dicionarios e acrescentar descritivos dos arquivos signals e raw_results

In [3]:
# Whether or not to create config
CREATE_CONFIG=True

# Whether or not to create folder structure
CREATE_FOLDER_STRUCTURE=True

# Experiment name
EXPERIMENT_NAME='profiling_test'

# Folder which will contain output directory tree
OUTPUT_DIR='./output'
base_output_path=Path(OUTPUT_DIR)

# Create experiment directory
experiment_dir = base_output_path / EXPERIMENT_NAME

# 1. Configurations

In [4]:
if CREATE_CONFIG:
    # Number of sources and observations
    NSOURCES=2
    NOBS=1000

    # Mixing matrix configuration
    A = np.array([
        [1, 1],
        [-0.5, 0.5]
    ])
    # A = np.array([
    #     [1/0.98, 0],
    #     [0, 0.98]
    # ])

    # Define initial conditions for optimizations
    initial_B = np.linalg.inv(
        A
    ) + np.random.normal(
        0,0.1,
        A.shape
    )
    initial_B = initial_B.reshape(
        (NSOURCES, NSOURCES, 1)
    )

    # Execution configurations
    exec_configs = {
        'general': {
            # Experiment name for folder structure
            'experiment_name': EXPERIMENT_NAME,
            # Experiment directory
            'experiment_dir': experiment_dir,
            # Whether or not to create folder structure
            'create_folder_structure': CREATE_FOLDER_STRUCTURE,
            # Whether or not to save graphs
            'save_graphs': False,
            # Number of sources
            'n_sources': NSOURCES,
            # Number of observations in each realization
            'n_obs': NOBS,
            # Number of parallel workers
            'n_workers': 1,
            # Number of parallel realizations to run
            'n_realizations': 1,
            # Mixing matrix
            'A': A,
            # Separating matrix
            'B': np.linalg.inv(A),
            # Initial condition
            'initial_B': initial_B,
            # Whether or not to use a normalized posterior
            'normalize_posterior': True
        },
        'contour': {
            # Number of points used in contour line grids
            'contour_grid_points': 301,
            # Exploration limits for grids
            'u_lims':(-0.3, 0.3),
            'v_lims':(-0.3, 0.3),
            'central_point': (0,0)
        },
        'map': {
            # Threshold for stopping optimization
            'stopping_thresh': 1E-7,
            # Maximum number of iterations
            'max_it': 10000000,
            # Learning rate
            'learning_rate': 1E-4,
            # Persistance iterations for stopping criterion
            'stopping_criterion_persistance_its': 20
        },
        'mcmc': {
            # Exploration variance for MCMC
            'exploration_var': 5E-5,
            # Number of samples to generate
            'n_samples': 30000,
            # Burn-in samples
            'burn_in': 0.5
        },
        'sim': {
            'source_model': LogisticSource(
                mu=0.0,
                sigma=1.0
            ),
            'sources': {
                'perfect_model': LogisticSource(
                    mu=0.0,
                    sigma=1.0
                ),
                'slightly_misspecified_model': LogisticSource(
                    mu=0.1,
                    sigma=1.1
                ),
                'largely_misspecified_model': TriangularSource(
                    lower=-2,
                    upper=2,
                    mode=0
                )
            },
            'priors': {
                'likelihood': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=np.inf
                ),
                'non_informative_prior': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=10
                ),
                'informative_prior': ExponentialPrior(
                    center=np.linalg.inv(A),
                    std=0.1
                ),
                'identity_transform': ExponentialPrior(
                    center=np.eye(NSOURCES),
                    std=0.1
                )
            },
            'test_cases': {
                'i': {
                    'source': 'perfect_model',
                    'prior': 'likelihood',
                },
                'ii': {
                    'source': 'perfect_model',
                    'prior': 'non_informative_prior',
                },
                'iii': {
                    'source': 'perfect_model',
                    'prior': 'informative_prior',
                },
                'iv': {
                    'source': 'perfect_model',
                    'prior': 'identity_transform',
                },
                'v': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'likelihood',
                },
                'vi': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'non_informative_prior',
                },
                'vii': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'informative_prior',
                },
                'viii': {
                    'source': 'slightly_misspecified_model',
                    'prior': 'identity_transform',
                },
                'ix': {
                    'source': 'largely_misspecified_model',
                    'prior': 'likelihood',
                },
                'x': {
                    'source': 'largely_misspecified_model',
                    'prior': 'non_informative_prior',
                },
                'xi': {
                    'source': 'largely_misspecified_model',
                    'prior': 'informative_prior',
                },
                'xii': {
                    'source': 'largely_misspecified_model',
                    'prior': 'identity_transform',
                },
            }
        }
    }
else:
    with (experiment_dir/'execution_config.pkl').open('rb') as f:
        exec_configs = dill.load(f)

pprint.pp(exec_configs)

{'general': {'experiment_name': 'profiling_test',
             'experiment_dir': PosixPath('output/profiling_test'),
             'create_folder_structure': True,
             'save_graphs': False,
             'n_sources': 2,
             'n_obs': 1000,
             'n_workers': 1,
             'n_realizations': 1,
             'A': array([[ 1. ,  1. ],
       [-0.5,  0.5]]),
             'B': array([[ 0.5, -1. ],
       [ 0.5,  1. ]]),
             'initial_B': array([[[ 0.67367376],
        [-0.81020861]],

       [[ 0.28932266],
        [ 0.98510879]]]),
             'normalize_posterior': True},
 'contour': {'contour_grid_points': 301,
             'u_lims': (-0.3, 0.3),
             'v_lims': (-0.3, 0.3),
             'central_point': (0, 0)},
 'map': {'stopping_thresh': 1e-07,
         'max_it': 10000000,
         'learning_rate': 0.0001,
         'stopping_criterion_persistance_its': 20},
 'mcmc': {'exploration_var': 5e-05, 'n_samples': 30000, 'burn_in': 0.5},
 'sim': {'source_

# 2. Folder Structure

In [5]:
# Initialize folder structure, if so specified
if CREATE_FOLDER_STRUCTURE:
    if exec_configs['general']['create_folder_structure']:
        # Creates base output path and experiment dir
        if not base_output_path.is_dir():
            base_output_path.mkdir()
        if not experiment_dir.is_dir():
            experiment_dir.mkdir()
        # Creates folders for individual realizations
        for r in range(exec_configs['general']['n_realizations']):
            # Overall folder for realization
            realization_dir = experiment_dir / str(r)
            if not realization_dir.is_dir():
                realization_dir.mkdir()
            # Initialize success flag
            with (realization_dir/'success_flag.pkl').open('wb') as f:
                    dill.dump('UNFINISHED', f)
        

if CREATE_CONFIG:
     with (experiment_dir/'execution_config.pkl').open('wb') as f:
        dill.dump(exec_configs, f)


# 3. Execute Profiler

In [6]:
# Define executor object
ex = ExperimentExecutor(
    cfg=exec_configs,
    initialize=CREATE_CONFIG
)

# Path for saving profiler stats
PROFILER_PATH = experiment_dir/'profiler_output'

# Execute profiler
cProfile.run(
    'ex.run()',
    filename=PROFILER_PATH.as_posix()
)

####################################################################################################
Total realizations: 1
Finished realizations: 0
Unfinished realizations: 1
####################################################################################################


In [ ]:
p = pstats.Stats(PROFILER_PATH.as_posix())
p.sort_stats(pstats.SortKey.TIME).print_stats()

Sun Jun 14 23:52:18 2026    output/profiling_test/profiler_output

         25971740022 function calls (25914970583 primitive calls) in 22855.960 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
5852844000 9740.325    0.000 9740.325    0.000 /Users/leonardo.tessarolo/git/bayesian_bss/src/source.py:16(__logistic_distribution)
766422000 3653.716    0.000 3653.716    0.000 /Users/leonardo.tessarolo/git/bayesian_bss/src/source.py:36(__logistic_distribution_derivative)
  1080000 1804.532    0.002 6364.050    0.006 /Users/leonardo.tessarolo/git/bayesian_bss/src/estimator.py:385(__log_posterior_fn)
  1080000 1788.704    0.002 6179.026    0.006 /Users/leonardo.tessarolo/git/bayesian_bss/src/contour_line.py:47(__log_posterior_fn)
     4372 1494.415    0.342 51194.606   11.710 /Users/leonardo.tessarolo/.pyenv/versions/3.13.5/lib/python3.13/asyncio/base_events.py:1962(_run_once)
5852844000  943.988    0.000 10684.313    0.000 /Users/le